##### Ingest pit_stops.json file

##### Read the JSON file using the spark dataframe reader API

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-28")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
pit_stops_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                      StructField("driverId", IntegerType(), True),
                                      StructField("stop", StringType(), True),
                                      StructField("lap", IntegerType(), True),
                                      StructField("time", StringType(), True),
                                      StructField("duration", StringType(), True),
                                      StructField("milliseconds", IntegerType(), True)
                                     ])

In [0]:
pit_stops_df = spark.read \
.schema(pit_stops_schema) \
.option("multiline", True) \
.json(f"{raw_path}/pit_stops.json") 

##### Rename columns and add new columns

In [0]:
final_df = pit_stops_df \
.withColumnRenamed("raceId", "race_id") \
.withColumnRenamed("driverId", "driver_id") \
.withColumn("data_source", lit(v_data_source))

In [0]:
final_df = add_ingestion_date(final_df)

##### Write file

In [0]:
# final_df.write.mode("overwrite").parquet(f"{processed_path}/pit_stops")

In [0]:
### final solution 
overwrite_partition_free(final_df, 'f1_processed', 'pit_stops', 'race_id')

##### Check

In [0]:
# spark.read.parquet(f"{processed_path}/pit_stops").display()

In [0]:
dbutils.notebook.exit("SUCCESS")

In [0]:
%sql
select race_id, count(1) as count
from f1_processed.pit_stops
group by race_id
order by race_id desc;